# Lectura de paquetes y data

In [12]:
import warnings
import os
import time
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna

# Agrega todo el directorio padre al path
sys.path.append(os.path.abspath(".."))
from src.utils_ml import ml_training_utils as ml_utils
from src.utils_ml import ml_feature_engineering as fe_utils

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

Utilizamos paths relativos para la lectura de la data

In [3]:
filename = "ml_pipeline_p_sku.ipynb"  # nombre del archivo actual
print(f"Current absolute path: {os.getcwd()}\n")

# Especificamos la ruta del directorio actual y los directorios de datos y salida
ACTUAL_DIR = os.path.dirname(os.path.abspath(filename))
BASE_DIR = os.path.dirname(ACTUAL_DIR)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

Current absolute path: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/notebooks

BASE_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi
DATA_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/data
OUTPUT_DIR: /Users/jhonattan.reales/Documents/Personal_jhonattan/Icesi/Innova_tec1/innv_tec_1_icesi/data/output


In [ ]:
# Cargar el archivo de Excel
file_path = os.path.join(DATA_DIR, "data_demanda.xlsx")
df_base = pd.read_excel(file_path, sheet_name="data")
df_base = df_base.drop("Cliente", axis=1)

df_base.shape

In [6]:
df_base.head(5)

,Fe.prefer.entrega,Day_of_the_Week,Pedidos,Sell_In,Sell_in_on_time,CEDIS,SKU,Mes_Año,Sell_in_rezago,porcentaje_on_time,Subcategoria
0,2024-07-05,Friday,16200,16200,16200,7,SKU7,72024,0,1.0,BEBIDAS DE YOGUR
1,2024-07-05,Friday,19800,19800,19800,6,SKU5,72024,0,1.0,BEBIDAS DE YOGUR
2,2024-07-05,Friday,4950,4950,4950,4,SKU6,72024,0,1.0,BEBIDAS DE YOGUR
3,2024-07-05,Friday,8100,8100,8100,6,SKU4,72024,0,1.0,BEBIDAS DE YOGUR
4,2024-07-05,Friday,18600,18600,18600,7,SKU3,72024,0,1.0,BEBIDAS DE YOGUR


In [9]:
df_base.tail(5)

,Fe.prefer.entrega,Day_of_the_Week,Pedidos,Sell_In,Sell_in_on_time,CEDIS,SKU,Mes_Año,Sell_in_rezago,porcentaje_on_time,Subcategoria
6854,2025-04-11,Friday,2400,2400,2400,2,SKU14,42025,0,1.0,FIOCOSSOS
6855,2025-04-11,Friday,5550,5565,5550,3,SKU16,42025,15,1.0,LECHE PASTEURIZADA
6856,2025-04-11,Friday,600,0,0,1,SKU8,42025,0,0.0,YOGUR BOTELLA
6857,2025-04-11,Friday,600,0,0,1,SKU22,42025,0,0.0,YOGUR VASOS
6858,2025-04-11,Friday,240,0,0,1,SKU15,42025,0,0.0,KUMIS


In [7]:
# Filtrar los datos relevantes para este analisis

df = (
    df_base[["Fe.prefer.entrega", "SKU", "Pedidos"]]
    .copy()
    .rename(
        columns={
            "Fe.prefer.entrega": "Fecha",
        }
    )
)
df["Pedidos"] = pd.to_numeric(df["Pedidos"], errors="coerce")

In [8]:
df

,Fecha,SKU,Pedidos
0,2024-07-05,SKU7,16200
1,2024-07-05,SKU5,19800
2,2024-07-05,SKU6,4950
3,2024-07-05,SKU4,8100
4,2024-07-05,SKU3,18600
...,...,...,...
6854,2025-04-11,SKU14,2400
6855,2025-04-11,SKU16,5550
6856,2025-04-11,SKU8,600
6857,2025-04-11,SKU22,600


# Preparación de la data

In [10]:
### Primero, nos aseguramos de que se cuente un dato por SKU por dia
# -------

# rango completo de fechas desde la más antigua hasta la más reciente
fecha_min = df["Fecha"].min()
fecha_max = df["Fecha"].max()
rango_fechas = pd.date_range(start=fecha_min, end=fecha_max, freq="D")

# Obtenemos todos los SKUs únicos
skus = df["SKU"].unique()

# DataFrame con todas las combinaciones de SKU y fecha
combinaciones_completas = pd.MultiIndex.from_product(
    [rango_fechas, skus], names=["Fecha", "SKU"]
).to_frame(index=False)

# Unir con el dataframe original para rellenar con ceros donde falten datos
df_completo = combinaciones_completas.merge(df, on=["Fecha", "SKU"], how="left")

# Rellenar valores faltantes de pedidos con 0
df_completo["Pedidos"] = df_completo["Pedidos"].fillna(0).astype(int)

# Ordenar por SKU y Fecha (opcional)
df_completo = df_completo.sort_values(["SKU", "Fecha"]).reset_index(drop=True)

df = df_completo.copy()

In [11]:
df

,Fecha,SKU,Pedidos
0,2024-07-05,SKU1,7200
1,2024-07-06,SKU1,7200
2,2024-07-07,SKU1,7200
3,2024-07-08,SKU1,3000
4,2024-07-09,SKU1,3000
...,...,...,...
7301,2025-04-07,SKU9,1200
7302,2025-04-08,SKU9,1200
7303,2025-04-09,SKU9,2760
7304,2025-04-10,SKU9,120


# Feature engineering

## Variables temporales

## Variables tipo lag

## Variables tipo promedio moviles

## Variables lags de STL

# Modelling 

## Evaluación modelos XGBoost

## Evaluación modelos Random Forest

## Selección mejor modelo y ajuste final

# Predicción y graficas 